In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

#Encoder (giả lập PointPillars output)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim)
        )

    def forward(self, x):
        # x: (B, D)
        return self.net(x)  # (B, latent_dim)

In [ ]:
class SimpleMamba(nn.Module):
    def __init__(self, latent_dim, hidden_dim, horizon):
        super().__init__()
        self.latent_dim = latent_dim
        self.horizon = horizon

        self.rnn = nn.GRU(latent_dim, hidden_dim, batch_first=True)
        self.proj = nn.Linear(hidden_dim, latent_dim)

    def forward(self, z_seq):
        # z_seq: (B, T, D)

        _, h = self.rnn(z_seq)  # h: (1, B, hidden)
        h = h.squeeze(0)

        outputs = []
        x = h

        for _ in range(self.horizon):
            x = torch.tanh(x)
            z_next = self.proj(x)
            outputs.append(z_next)

        z_future = torch.stack(outputs, dim=1)  # (B, H, D)
        return z_future

In [ ]:
class RiskHead(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, z_future):
        # z_future: (B, H, D)
        z = z_future.mean(dim=1)  # (B, D)
        risk = torch.sigmoid(self.net(z))
        return risk

In [ ]:
class WorldModel(nn.Module):
    def __init__(self, input_dim, latent_dim, hidden_dim, horizon):
        super().__init__()
        self.encoder = Encoder(input_dim, latent_dim)
        self.mamba = SimpleMamba(latent_dim, hidden_dim, horizon)
        self.risk_head = RiskHead(latent_dim)

    def forward(self, obs_seq):
        # obs_seq: (B, T, input_dim)

        B, T, D = obs_seq.shape

        z_seq = []
        for t in range(T):
            z = self.encoder(obs_seq[:, t])
            z_seq.append(z)

        z_seq = torch.stack(z_seq, dim=1)  # (B, T, latent)

        z_future = self.mamba(z_seq)
        risk = self.risk_head(z_future)

        return z_future, risk

In [ ]:
# LOSS FUNCTION

def compute_loss(model, batch, alpha=1.0):
    obs_seq = batch["obs_seq"]          # (B, T, D)
    future_obs = batch["future_obs"]    # (B, H, D)
    risk_gt = batch["risk"]             # (B, 1)

    z_future_pred, risk_pred = model(obs_seq)

    # ===== TRAJECTORY LOSS =====
    with torch.no_grad():
        z_future_gt = []
        for t in range(future_obs.shape[1]):
            z = model.encoder(future_obs[:, t])
            z_future_gt.append(z)
        z_future_gt = torch.stack(z_future_gt, dim=1)

    L_traj = F.mse_loss(z_future_pred, z_future_gt)

    # ===== RISK LOSS =====
    risk_gt = risk_gt.unsqueeze(1)
    L_risk = F.binary_cross_entropy(risk_pred, risk_gt)

    # ===== TOTAL =====
    loss = L_traj + alpha * L_risk

    return loss, {
        "traj": L_traj.item(),
        "risk": L_risk.item()
    }

In [ ]:
#TRAINING LOOP (PRETRAIN)

def train(model, dataloader, epochs=10, lr=1e-3, device="cuda"):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        total_loss = 0

        for batch in dataloader:
            obs_seq = batch["obs_seq"].to(device)
            future_obs = batch["future_obs"].to(device)
            risk = batch["risk"].to(device)

            loss, logs = compute_loss(
                model,
                {
                    "obs_seq": obs_seq,
                    "future_obs": future_obs,
                    "risk": risk
                }
            )

            optimizer.zero_grad()
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch} | Loss: {total_loss:.4f} | "
              f"traj: {logs['traj']:.4f} risk: {logs['risk']:.4f}")

In [ ]:
#Dataloader example

from torch.utils.data import DataLoader

dataset = TrajDataset(data)  # từ builder trước
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
# Run train

model = WorldModel(
    input_dim=20,   # tùy dataset bạn
    latent_dim=64,
    hidden_dim=128,
    horizon=5
)

train(model, loader, epochs=20)

Model học được:
- Encoder
compress observation
- Mamba
hiểu dynamics
- Risk head
predict collision future

In [ ]:
#Phần sau sẽ nối vào RL
z_future, risk = model(obs_seq)
action = policy(torch.cat([z_t, risk], dim=-1))

#--------------

# đúng
risk = risk_head(z_future)

# sai
risk = risk_head(z_future).detach()
